# OOD testing: stand-in players that differ from their history

**Milestone:** "Out-of-distribution testing: measure how the player model behaves against
stand-in players whose style differs from their history, and against players much stronger or
weaker than estimated" (README Teaching). Also the README claim: *the engine never breaks on odd
moves; what can fail is the player model.*

Stand-in players are **engine personas**: an engine plus rules that give a style and a strength.
Because we control them, we know the ground truth the model should find.

## Experiment plan

| # | History | Test games | Expected model behaviour |
|---|---|---|---|
| E1 | persona A (N games) | persona A | estimate converges, confidence rises, OOD low |
| E2 | persona A | persona B (different style, same strength) | OOD rises; teacher falls back |
| E3 | persona A | A at +400 Elo (much stronger) | OOD rises fast; no exploitation of stale habits |
| E4 | persona A | A at -400 Elo (much weaker / rusty) | OOD rises; fallback at last estimated level |
| E5 | persona A | A "hiding strength" (plays weak early, strong later) | live OOD fires mid-game |

Measured per experiment: Elo estimate error, CI coverage, confidence, OOD score per game,
teacher fallback rate (`TeacherDecision.reason`), balanced share (metrics.ipynb).

**TODO checklist**
- [x] Persona data type, a noisy-engine persona, game harness, model evaluation.
- [ ] Style personas (`aggressive`, `passive`, `tactically blind`, `opening swapper`, `strength hider`).
- [ ] Strength anchors: Stockfish UCI_Elo personas, or human-like engines (Maia) for realism.
- [ ] Run E1..E5 in `run_teaching_experiment.ipynb` and record results in the Wiki.

In [ ]:
%run ./move_selection.ipynb

In [ ]:
import random
from dataclasses import dataclass, field
from typing import Optional

import chess
import chess.pgn

## Personas

In [ ]:
@dataclass
class Persona:
    name: str
    description: str
    target_elo: Optional[int] = None      # for UCI personas (Stockfish UCI_Elo)
    blunder_rate: float = 0.0             # chance of a uniformly random move
    style: str = "neutral"                # neutral | aggressive | passive | tactically_blind | ...


class PersonaPlayer:
    """An engine move most of the time, a random move with probability `blunder_rate`."""

    def __init__(self, persona: Persona, engine: EngineLike, seed: int = 0):
        self.persona, self.engine, self.rng = persona, engine, random.Random(seed)

    def play(self, board: chess.Board) -> chess.Move:
        if self.persona.style != "neutral":
            return style_move(self.persona, board, self.engine, self.rng)
        if self.rng.random() < self.persona.blunder_rate:
            return self.rng.choice(list(board.legal_moves))
        return self.engine.play(board)


def style_move(persona: Persona, board: chess.Board, engine: EngineLike, rng: random.Random) -> chess.Move:
    """Move for a styled persona.

    TODO: e.g. `aggressive`: among multipv candidates within 100 cp of the best, prefer captures,
    checks and pawn storms; `passive`: prefer quiet moves and trades; `tactically_blind`: never play
    the best move when it is forcing; `strength_hider`: blunder_rate high for 15 moves, then 0.
    """
    raise NotImplementedError(f"style persona '{persona.style}'")


PERSONAS = {
    "solid": Persona("solid", "engine moves, rare random blunder", blunder_rate=0.05),
    "sloppy": Persona("sloppy", "engine moves, frequent random blunders (rusty player)", blunder_rate=0.3),
    # TODO: styled personas, see style_move
    "aggressive": Persona("aggressive", "prefers captures and checks", style="aggressive"),
    "tactically_blind": Persona("tactically_blind", "misses forcing moves", style="tactically_blind"),
}

## Harness: play a persona against a policy (software only, no physical board)

In [ ]:
def play_persona_game(persona_player: PersonaPlayer, robot,  # any object with choose_move(board) (game_manager.MovePolicy)
                      persona_color: chess.Color = chess.WHITE,
                      max_plies: int = 200, game_id: str = "ood") -> chess.pgn.Game:
    board = chess.Board()
    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:
        mv = persona_player.play(board) if board.turn == persona_color else robot.choose_move(board)
        board.push(mv)
    game = chess.pgn.Game.from_board(board)
    name = persona_player.persona.name
    game.headers.update({"Event": game_id, "Site": game_id,
                         "White": name if persona_color == chess.WHITE else "Gambit",
                         "Black": "Gambit" if persona_color == chess.WHITE else name})
    return game


def evaluate_model_on_games(model: PlayerModel, games: list, analyst: EngineLike, color: chess.Color,
                            true_elo: Optional[float] = None, update: bool = True) -> list[dict]:
    """For each game: OOD score vs the history *before* the game, then (optionally) update."""
    rows = []
    for i, g in enumerate(games):
        analyses = analyse_game(g, analyst, color, f"{g.headers.get('Event')}-{i}")
        if not analyses:
            continue
        summary = summarize_game(analyses)
        ood = model.ood_score(summary)
        if update:
            model.add_game_summary(summary)
        est = model.estimate()
        rows.append({"game": i, "acpl": summary.acpl, "ood_score": ood, "elo_est": est.elo,
                     "elo_low": est.elo_low, "elo_high": est.elo_high, "confidence": est.confidence,
                     "elo_error": (est.elo - true_elo) if true_elo is not None else None,
                     "ci_covers": (est.elo_low <= true_elo <= est.elo_high) if true_elo is not None else None})
    return rows

## Self-check: tiny run (short games, depth-1 engines)

In [ ]:
fast = MaterialEngine(depth=1)
solid = PersonaPlayer(PERSONAS["solid"], fast, seed=1)
sloppy = PersonaPlayer(PERSONAS["sloppy"], fast, seed=2)


class _EnginePolicy:                       # same as game_manager.EnginePolicy, without the %run
    def __init__(self, e): self.e = e
    def choose_move(self, b): return self.e.play(b)

robot = _EnginePolicy(fast)
history = [play_persona_game(solid, robot, max_plies=16, game_id="solid") for _ in range(3)]
assert all(len(list(g.mainline_moves())) == 16 for g in history)
model = PlayerModel("solid-persona")
rows = evaluate_model_on_games(model, history, fast, chess.WHITE)
assert len(rows) == 3 and math.isinf(rows[0]["ood_score"]) and len(model.games) == 3
test = evaluate_model_on_games(model, [play_persona_game(sloppy, robot, max_plies=16, game_id="sloppy")],
                               fast, chess.WHITE, update=False)
assert math.isfinite(test[0]["ood_score"]) and len(model.games) == 3
try:
    PersonaPlayer(PERSONAS["aggressive"], fast).play(chess.Board())
    raise AssertionError("styled personas should be TODO")
except NotImplementedError:
    pass
print("ood_testing self-check passed")